In [6]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from datetime import datetime
import warnings
import os
import re
warnings.filterwarnings('ignore')

print("="*70)
print("CRYPTOCURRENCY NEWS SENTIMENT ANALYSIS WITH FINBERT")
print("="*70)

# ============================================================
# 1. LOAD AND PREPARE DATA
# ============================================================
print("\n📊 Loading news headlines data...")

# Load the CSV file
df = pd.read_csv('CryptoDataSet.csv')
print(f"✓ Loaded {len(df):,} news articles")
print(f"   Columns: {df.columns.tolist()}")

# Clean the data
print("\n🧹 Cleaning data...")
initial_rows = len(df)

# Drop rows with missing dates
df = df.dropna(subset=['Date Time'])
print(f"   Removed {initial_rows - len(df)} rows with missing dates")

# Convert datetime and handle errors
df['Date Time'] = pd.to_datetime(df['Date Time'], errors='coerce')
df = df.dropna(subset=['Date Time'])
df['date_only'] = df['Date Time'].dt.date

# Clean titles - remove problematic characters
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove or replace problematic characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
    return text.strip()

df['clean_title'] = df['Title'].apply(clean_text)

# Remove empty titles
df = df[df['clean_title'] != '']
df = df.dropna(subset=['clean_title'])

print(f"✓ Cleaned data: {len(df):,} valid news articles")
print(f"📅 Date range: {df['date_only'].min()} to {df['date_only'].max()}")
print(f"   Coin types: {df['Coin Type'].unique().tolist()}")

# ============================================================
# 2. FINBERT SENTIMENT ANALYSIS
# ============================================================
print("\n🧠 Loading FinBERT model for sentiment analysis...")

# Load FinBERT model and tokenizer
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print(f"✓ Model loaded on {device}")

def get_finbert_sentiment(texts, batch_size=32):
    """
    Get sentiment scores using FinBERT with better error handling
    """
    results = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Processing with FinBERT"):
        batch_texts = texts[i:i+batch_size]
        
        # Clean batch texts
        clean_batch = []
        for text in batch_texts:
            if pd.isna(text) or text == '':
                clean_batch.append("")
            else:
                # Ensure text is string and truncate if too long
                text = str(text)[:1000]  # FinBERT can handle up to 512 tokens, but we'll be safe
                clean_batch.append(text)
        
        try:
            # Tokenize with error handling
            inputs = tokenizer(clean_batch, padding=True, truncation=True, 
                              return_tensors="pt", max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Get predictions
            with torch.no_grad():
                outputs = model(**inputs)
                probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
                
            # Convert to numpy
            probs = probabilities.cpu().numpy()
            
            # Process each result
            for prob in probs:
                neg_score = prob[0]
                neutral_score = prob[1]
                pos_score = prob[2]
                
                # Determine label
                if pos_score > neg_score and pos_score > neutral_score:
                    label = 'POSITIVE'
                    score = pos_score
                elif neg_score > pos_score and neg_score > neutral_score:
                    label = 'NEGATIVE'
                    score = neg_score
                else:
                    label = 'NEUTRAL'
                    score = neutral_score
                
                # Create compound-like score (-1 to 1)
                compound = pos_score - neg_score
                
                results.append({
                    'label': label,
                    'confidence': float(score),
                    'compound': float(compound),
                    'positive': float(pos_score),
                    'negative': float(neg_score),
                    'neutral': float(neutral_score)
                })
                
        except Exception as e:
            print(f"\n⚠️ Error processing batch {i}-{i+batch_size}: {e}")
            # Append default values for failed batch
            for _ in range(len(batch_texts)):
                results.append({
                    'label': 'NEUTRAL',
                    'confidence': 0.5,
                    'compound': 0.0,
                    'positive': 0.33,
                    'negative': 0.33,
                    'neutral': 0.34
                })
    
    return results

# ============================================================
# 3. APPLY FINBERT TO NEWS TITLES
# ============================================================
print("\n🔍 Analyzing sentiment for news headlines...")

# Use a smaller sample for testing (remove this line for full processing)
# df = df.head(1000)  # Uncomment to test with smaller sample

# Process titles in batches
sentiment_results = get_finbert_sentiment(df['clean_title'].tolist())

# Verify we have results for all rows
print(f"✓ Processed {len(sentiment_results)} headlines")
print(f"   Expected: {len(df)}")

if len(sentiment_results) != len(df):
    print(f"⚠️ Mismatch: {len(sentiment_results)} results for {len(df)} rows")
    # Pad or truncate to match
    if len(sentiment_results) < len(df):
        # Add default values for missing results
        for _ in range(len(df) - len(sentiment_results)):
            sentiment_results.append({
                'label': 'NEUTRAL',
                'confidence': 0.5,
                'compound': 0.0,
                'positive': 0.33,
                'negative': 0.33,
                'neutral': 0.34
            })
    else:
        # Truncate extra results
        sentiment_results = sentiment_results[:len(df)]

# Add results to dataframe
df['finbert_label'] = [r['label'] for r in sentiment_results]
df['finbert_confidence'] = [r['confidence'] for r in sentiment_results]
df['finbert_compound'] = [r['compound'] for r in sentiment_results]
df['finbert_positive'] = [r['positive'] for r in sentiment_results]
df['finbert_negative'] = [r['negative'] for r in sentiment_results]
df['finbert_neutral'] = [r['neutral'] for r in sentiment_results]

# Show sample
print("\n📝 Sample sentiment results:")
sample = df[['Title', 'finbert_label', 'finbert_compound', 'finbert_confidence']].head(10)
print(sample.to_string())

# Show distribution
print("\n📊 Sentiment Distribution:")
print(df['finbert_label'].value_counts())
print(f"Average compound score: {df['finbert_compound'].mean():.4f}")

# Save detailed results
df.to_csv('news_with_finbert_sentiment.csv', index=False)
print("\n💾 Saved detailed results to: news_with_finbert_sentiment.csv")

# ============================================================
# 4. CALCULATE WEIGHTED DAILY SENTIMENT
# ============================================================
print("\n📈 Calculating weighted daily sentiment...")

def calculate_weighted_sentiment(df, coin_type=None):
    """
    Calculate daily weighted sentiment for a specific coin type
    """
    # Filter by coin type if specified
    if coin_type:
        coin_df = df[df['Coin Type'] == coin_type].copy()
    else:
        coin_df = df.copy()
        coin_type = 'ALL'
    
    if len(coin_df) == 0:
        print(f"⚠️ No data for {coin_type}")
        return pd.DataFrame()
    
    # Create engagement weight based on price movement and volume
    coin_df['abs_price_change'] = abs(coin_df['Movement_OpenClose_%'].fillna(0))
    coin_df['abs_price_range'] = abs(coin_df['Movement_HighLow_%'].fillna(0))
    
    # Handle volume - fill NaN with median
    volume_median = coin_df['Volume'].median()
    coin_df['Volume'] = coin_df['Volume'].fillna(volume_median)
    
    # Normalize volume
    if coin_df['Volume'].max() > coin_df['Volume'].min():
        coin_df['volume_norm'] = (coin_df['Volume'] - coin_df['Volume'].min()) / (coin_df['Volume'].max() - coin_df['Volume'].min() + 1)
    else:
        coin_df['volume_norm'] = 0
    
    # Combine into engagement weight
    coin_df['engagement_weight'] = (
        coin_df['abs_price_change'] * 2 +
        coin_df['volume_norm'] * 1 +
        (coin_df['abs_price_range'] * 1.5)
    )
    coin_df['engagement_weight'] = coin_df['engagement_weight'].clip(lower=0)
    
    # Weighted sentiment
    coin_df['weighted_sentiment'] = coin_df['finbert_compound'] * coin_df['engagement_weight']
    
    # Group by date
    daily = coin_df.groupby('date_only').agg({
        'finbert_compound': ['mean', 'std', 'count'],
        'finbert_positive': 'mean',
        'finbert_negative': 'mean',
        'finbert_neutral': 'mean',
        'weighted_sentiment': 'sum',
        'engagement_weight': 'sum',
        'abs_price_change': 'mean',
        'Volume': 'mean',
        'Movement_OpenClose_%': 'mean',
        'Market_Move': lambda x: (x == 'Up').sum() / len(x) if len(x) > 0 else 0
    }).round(4)
    
    # Flatten columns
    daily.columns = [
        'sentiment_mean', 'sentiment_std', 'news_count',
        'positive_mean', 'negative_mean', 'neutral_mean',
        'weighted_sentiment_sum', 'total_engagement_weight',
        'avg_price_change', 'avg_volume', 'avg_movement',
        'up_ratio'
    ]
    
    # Calculate weighted average sentiment
    daily['weighted_sentiment_mean'] = daily.apply(
        lambda row: row['weighted_sentiment_sum'] / row['total_engagement_weight'] 
        if row['total_engagement_weight'] > 0 else row['sentiment_mean'],
        axis=1
    ).round(4)
    
    # Calculate sentiment ratio
    daily['pos_neg_ratio'] = (
        daily['positive_mean'] / (daily['negative_mean'] + 0.0001)
    ).round(2)
    
    # Sentiment category
    daily['sentiment_category'] = pd.cut(
        daily['weighted_sentiment_mean'],
        bins=[-1, -0.3, -0.1, 0.1, 0.3, 1],
        labels=['Very Negative', 'Negative', 'Neutral', 'Positive', 'Very Positive']
    )
    
    # Reset index
    daily_sentiment = daily.reset_index()
    daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date_only']).dt.date
    
    # Keep relevant columns
    daily_sentiment = daily_sentiment[[
        'date', 'news_count', 'avg_volume', 'total_engagement_weight',
        'sentiment_mean', 'weighted_sentiment_mean', 'sentiment_std',
        'positive_mean', 'negative_mean', 'neutral_mean',
        'pos_neg_ratio', 'sentiment_category', 'avg_price_change', 'up_ratio'
    ]]
    
    return daily_sentiment

# ============================================================
# 5. GENERATE DAILY SENTIMENT FOR EACH COIN
# ============================================================
print("\n📊 Generating daily sentiment reports...")

# Get unique coin types
coin_types = df['Coin Type'].unique()
print(f"Found {len(coin_types)} coin types: {coin_types[:5]}...")

# Generate for each coin type
for coin in coin_types[:5]:  # Limit to first 5 for testing, remove [:5] for all
    print(f"\n🪙 Processing {coin}...")
    daily_df = calculate_weighted_sentiment(df, coin)
    
    if not daily_df.empty:
        # Save to CSV
        filename = f"news_sentiment_{coin.lower().replace(' ', '_')}_daily.csv"
        daily_df.to_csv(filename, index=False)
        print(f"✅ Saved {len(daily_df)} days to {filename}")
        
        # Show sample
        print(f"   Sample data:")
        print(daily_df.head(3).to_string())

# Also generate for all coins combined
print(f"\n🪙 Processing ALL coins combined...")
daily_all = calculate_weighted_sentiment(df)
if not daily_all.empty:
    filename = "news_sentiment_all_daily.csv"
    daily_all.to_csv(filename, index=False)
    print(f"✅ Saved {len(daily_all)} days to {filename}")

# ============================================================
# 6. SUMMARY STATISTICS
# ============================================================
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

summary_data = []
for coin in coin_types[:5]:  # Match the coins we processed
    filename = f"news_sentiment_{coin.lower().replace(' ', '_')}_daily.csv"
    try:
        if os.path.exists(filename):
            df_sum = pd.read_csv(filename)
            df_sum['date'] = pd.to_datetime(df_sum['date'])
            
            summary_data.append({
                'Coin': coin,
                'Days': len(df_sum),
                'Date Range': f"{df_sum['date'].min().date()} to {df_sum['date'].max().date()}",
                'Avg Sentiment': f"{df_sum['weighted_sentiment_mean'].mean():.4f}",
                'Positive Days': f"{(len(df_sum[df_sum['weighted_sentiment_mean'] > 0.05]) / len(df_sum) * 100):.1f}%",
                'Negative Days': f"{(len(df_sum[df_sum['weighted_sentiment_mean'] < -0.05]) / len(df_sum) * 100):.1f}%",
                'Avg News/Day': f"{df_sum['news_count'].mean():.1f}",
                'Total News': f"{df_sum['news_count'].sum():,}"
            })
    except Exception as e:
        print(f"⚠️ Could not read {filename}: {e}")

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n", summary_df.to_string(index=False))
    
    # Save summary
    summary_df.to_csv('news_sentiment_summary.csv', index=False)
    print("\n💾 Saved summary to: news_sentiment_summary.csv")

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE")
print("="*70)
print("\nGenerated files:")
for coin in coin_types[:5]:
    filename = f"news_sentiment_{coin.lower().replace(' ', '_')}_daily.csv"
    if os.path.exists(filename):
        print(f"   • {filename}")
if os.path.exists('news_sentiment_all_daily.csv'):
    print("   • news_sentiment_all_daily.csv")
if os.path.exists('news_sentiment_summary.csv'):
    print("   • news_sentiment_summary.csv")
print("="*70)

CRYPTOCURRENCY NEWS SENTIMENT ANALYSIS WITH FINBERT

📊 Loading news headlines data...
✓ Loaded 188,431 news articles
   Columns: ['URL', 'Title', 'Date Time', 'Coin Type', 'sentiment_label', 'sentiment_score', 'Open', 'High', 'Low', 'Close', 'Volume', 'Movement_OpenClose_%', 'Movement_HighLow_%', 'Market_Move']

🧹 Cleaning data...
   Removed 3 rows with missing dates
✓ Cleaned data: 188,105 valid news articles
📅 Date range: 2017-08-17 to 2025-08-27
   Coin types: ['Bitcoin', 'Ethereum', 'Litecoin', 'Binance Coin', 'Cardano', 'Ripple', 'XRP', 'VeChain', 'Chainlink', 'Dogecoin', 'Solana', 'Polkadot', 'Polygon', 'Uniswap', 'Terra', 'Avalanche', 'Shiba Inu']

🧠 Loading FinBERT model for sentiment analysis...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded on cpu

🔍 Analyzing sentiment for news headlines...


Processing with FinBERT:   0%|                 | 3/5879 [00:01<36:12,  2.70it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Processing with FinBERT: 100%|██████████████| 5879/5879 [42:24<00:00,  2.31it/s]


✓ Processed 188105 headlines
   Expected: 188105

📝 Sample sentiment results:
                                                                                                        Title finbert_label  finbert_compound  finbert_confidence
0                                              Anonymous Email Service ProtonMail Adds Bitcoin Payment Option      POSITIVE          0.901205            0.939015
1         Blockchain & Bitcoin Conference Stockholm to Feature Discussions on ICOs and Blockchain Development      POSITIVE          0.874801            0.931154
2                                                       Bitcoin Prices Reach New All-Time High of Over $4,500      NEGATIVE         -0.727750            0.853082
3                                   Lightning Bank Ledgers? Bitfury and Ripple Demo New Twist on Bitcoin Tech      POSITIVE          0.680692            0.826574
4                                           $7 Million: Bitcoin Wallet Startup Breadwallet Raises New Funding   